## 🎯 Learning Objectives
* Understand the concept of distributed agents and the actor model.
* Learn how AutoGen agents can be conceptualized as actors in a distributed system.
* Implement a basic simulation of distributed AutoGen agents using Python's multiprocessing module.
* Identify the benefits and trade-offs of distributing AI agents.
* Explore common use cases for distributed agent systems.


## Distributed Agents with AutoGen's Actor Model

In the rapidly evolving landscape of AI, the complexity and scale of tasks demand more than just a single, monolithic agent. Imagine a bustling city where every service, from traffic control to emergency response, is handled by a single, overwhelmed entity. This is analogous to a single AI agent trying to tackle all aspects of a complex problem. The solution lies in **distributed systems**, where multiple specialized agents collaborate, each handling a specific part of the overall challenge.

### The Actor Model: A Paradigm for Distribution

The **Actor Model** is a conceptual model for concurrent computation that treats "actors" as the universal primitives of computation. In this model, every component is an actor, and actors communicate *only* by sending and receiving messages. They never share memory or state directly. This paradigm offers several key advantages:

1.  **Isolation**: Each actor manages its own internal state, preventing race conditions and simplifying concurrency.
2.  **Concurrency**: Actors can execute in parallel, significantly speeding up complex tasks.
3.  **Resilience**: If one actor fails, it doesn't necessarily bring down the entire system. Other actors can continue their work.
4.  **Scalability**: New actors can be added or removed dynamically, allowing systems to scale horizontally across multiple cores, machines, or even data centers.

Think of actors as highly specialized, independent workers in a factory. Each worker (actor) has their own workbench (state), tools (capabilities), and a mailbox (message queue). When a worker needs something done, they don't directly manipulate another worker's tools; instead, they write a request on a note (message) and put it in the other worker's mailbox. The receiving worker processes the note at their own pace and, if necessary, sends a response back. This clear separation of concerns and asynchronous communication is the essence of the actor model.

### AutoGen Agents as Actors

AutoGen's `ConversableAgent` naturally aligns with the actor model. Each AutoGen agent:

*   **Has its own state**: This includes its `system_message`, `llm_config`, registered functions, and conversation history.
*   **Communicates via messages**: Agents interact by sending messages to each other, initiating chats, and replying to prompts.
*   **Performs actions**: Agents can execute code, call tools, or generate LLM responses based on received messages.
*   **Operates asynchronously**: While AutoGen's default `initiate_chat` is synchronous from the caller's perspective, the underlying LLM calls and tool executions are often asynchronous, and agents can be designed to handle messages without blocking the entire system.

By leveraging Python's `multiprocessing` module or dedicated distributed computing frameworks like Ray, we can run AutoGen agents in separate processes or even on different machines. This allows us to build truly distributed multi-agent systems where each agent acts as an independent actor, communicating via explicit message passing. This approach is crucial for building robust, scalable, and fault-tolerant AI applications in 2026 and beyond.


In [ ]:
import autogen
import multiprocessing
import time
import json
import os

# --- Configuration for AutoGen Agents ---
# IMPORTANT: Ensure your OAI_CONFIG_LIST environment variable or file is set up.
# For demonstration, we'll use a placeholder. In a real scenario, this would
# point to your actual LLM API keys and models.
# Example OAI_CONFIG_LIST.json content:
# [
#     {
#         "model": "gpt-4o",
#         "api_key": "YOUR_OPENAI_API_KEY"
#     }
# ]

# Fallback for demonstration if OAI_CONFIG_LIST is not set
if "OAI_CONFIG_LIST" not in os.environ:
    print("Warning: OAI_CONFIG_LIST environment variable not found. Using dummy config.")
    # This dummy config will likely fail if no actual API key is provided.
    # Replace with your actual configuration for a runnable example.
    config_list = [
        {
            "model": "gpt-4o",
            "api_key": "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
        }
    ]
else:
    config_list = autogen.config_list_from_json(
        "OAI_CONFIG_LIST",
        filter_dict={
            "model": ["gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"], # Modern models
        },
    )

llm_config = {"config_list": config_list, "temperature": 0.7}

# --- Remote Assistant Agent Worker Function ---
# This function runs in a separate process and hosts an AutoGen AssistantAgent.
# It communicates with the main process via multiprocessing.Queue.
def remote_assistant_worker(input_queue, output_queue, worker_llm_config):
    """
    Function to be run in a separate process, hosting an AutoGen AssistantAgent.
    It listens for tasks on input_queue and sends responses to output_queue.
    """
    pid = multiprocessing.current_process().pid
    print(f"[Worker {pid}] Assistant worker process started.")

    # Define a custom function for the assistant to "perform" a task.
    # This simulates a specialized capability of a remote agent.
    def perform_complex_calculation(data: str) -> str:
        """
        Performs a complex mathematical calculation on the input data.
        """
        print(f"[Worker {pid}] Assistant is performing calculation on: {data}")
        try:
            num = float(data)
            # Simulate a computationally intensive task
            time.sleep(2) 
            result = num * num + (num / 2) - 10 # Example complex calculation
            return f"The result of the complex calculation for {data} is {result:.2f}."
        except ValueError:
            return f"Invalid input for calculation: {data}"

    # Create the AssistantAgent within this worker process.
    assistant = autogen.AssistantAgent(
        name="RemoteAssistant",
        llm_config=worker_llm_config,
        system_message="You are a helpful assistant capable of performing complex calculations. If asked to calculate, use the 'perform_complex_calculation' function. Respond concisely.",
    )

    # Register the custom function. The LLM can now decide to call this function.
    assistant.register_for_execution(perform_complex_calculation)
    assistant.register_for_llm(perform_complex_calculation)

    # Create a temporary UserProxyAgent within the worker to initiate chat with the Assistant.
    # This allows the AssistantAgent to use its full conversational logic, including function calling,
    # based on messages received from the main process via the queue.
    temp_user_proxy = autogen.UserProxyAgent(
        name="TempUserProxy",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False, # No code execution needed for this proxy
        is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
        system_message="You are a proxy for the main process. Relay messages to the RemoteAssistant and send its final reply back."
    )

    while True:
        try:
            # Wait for a message (task) from the main process.
            message = input_queue.get()
            if message == "STOP":
                print(f"[Worker {pid}] Assistant worker process received STOP signal.")
                break

            print(f"[Worker {pid}] RemoteAssistant received task: '{message}'")

            # Initiate a chat with the AssistantAgent using the received message.
            # The AssistantAgent will process this message, potentially calling its registered function.
            temp_user_proxy.initiate_chat(
                assistant,
                message=message,
            )

            # Get the last message from the assistant, which is its final response.
            response = assistant.last_message(assistant)
            if response:
                response_content = response.get("content", "No content")
                print(f"[Worker {pid}] RemoteAssistant processed task, response: '{response_content}'")
                output_queue.put(response_content)
            else:
                output_queue.put("Error: Assistant did not provide a response.")

        except Exception as e:
            print(f"[Worker {pid}] Assistant worker process error: {e}")
            output_queue.put(f"Error processing task: {e}")

# --- Main Process (User Proxy) ---
# This function orchestrates the interaction with the remote assistant worker.
def main_distributed_chat():
    print("\n[Main Process] Main process started.")

    # Queues for inter-process communication (mimicking message mailboxes).
    # input_queue: Main process -> Assistant worker
    # output_queue: Assistant worker -> Main process
    input_queue = multiprocessing.Queue()
    output_queue = multiprocessing.Queue()

    # Start the remote assistant worker process.
    # The worker_llm_config is passed to ensure the assistant in the new process
    # has its own LLM configuration.
    assistant_process = multiprocessing.Process(
        target=remote_assistant_worker,
        args=(input_queue, output_queue, llm_config)
    )
    assistant_process.start()
    print(f"[Main Process] Assistant worker process started with PID: {assistant_process.pid}")

    try:
        # --- Task 1: Complex Calculation ---
        task_message_1 = "Please perform a complex calculation on the number 123.45."
        print(f"\n[Main Process] Sending task to remote assistant: '{task_message_1}'")
        input_queue.put(task_message_1)

        # Wait for response from the remote assistant.
        response_1 = output_queue.get(timeout=90) # Increased timeout for LLM calls
        print(f"[Main Process] Received response from remote assistant: '{response_1}'")

        # --- Task 2: General Query ---
        task_message_2 = "What is the capital of France?"
        print(f"\n[Main Process] Sending general query to remote assistant: '{task_message_2}'")
        input_queue.put(task_message_2)

        response_2 = output_queue.get(timeout=90)
        print(f"[Main Process] Received response from remote assistant: '{response_2}'")

        # --- Task 3: Another Calculation ---
        task_message_3 = "Calculate for 789.01."
        print(f"\n[Main Process] Sending another calculation task: '{task_message_3}'")
        input_queue.put(task_message_3)

        response_3 = output_queue.get(timeout=90)
        print(f"[Main Process] Received response from remote assistant: '{response_3}'")

    except Exception as e:
        print(f"\n[Main Process] Error during interaction: {e}")
    finally:
        # Signal the worker to stop gracefully.
        print("\n[Main Process] Sending STOP signal to assistant worker.")
        input_queue.put("STOP")
        assistant_process.join(timeout=15) # Wait for the worker to terminate
        if assistant_process.is_alive():
            print("[Main Process] Assistant worker did not terminate gracefully, terminating forcefully.")
            assistant_process.terminate()
        print("[Main Process] Main process finished.")

# Entry point for the script
if __name__ == '__main__':
    # This ensures that the code inside main_distributed_chat() runs only when
    # the script is executed directly, and not when imported by multiprocessing.
    main_distributed_chat()


### Interpreting the Code Output and Performance Trade-offs

When you run the code, you'll observe output from two distinct processes:

*   **`[Main Process]`**: This represents your primary application or a coordinating agent. It initiates tasks and collects results.
*   **`[Worker PID]`**: This represents the remote agent, running in a separate Python process. It receives tasks, processes them (potentially involving LLM calls and tool execution), and sends back results.

You'll see messages flowing from the main process to the worker's input queue, and then responses flowing back from the worker's output queue. The `RemoteAssistant` in the worker process uses its `perform_complex_calculation` function when prompted, demonstrating how a specialized capability can be encapsulated and executed remotely. For general queries, it leverages its LLM capabilities.

This setup directly illustrates the **actor model**:

*   Each `multiprocessing.Process` acts as an independent **actor**.
*   `multiprocessing.Queue` instances serve as the **message mailboxes** for inter-actor communication.
*   Messages are immutable data sent between actors, avoiding shared state.

#### Performance Trade-offs and Considerations:

1.  **Inter-Process Communication (IPC) Overhead**: Sending data between processes involves serialization (e.g., pickling Python objects) and deserialization. For large messages or very frequent communication, this can introduce significant overhead, impacting latency and throughput.
2.  **Resource Consumption**: Each process has its own memory space and Python interpreter. Running many agents in separate processes can consume substantial CPU and RAM, especially if each agent requires its own LLM context or large models.
3.  **Complexity of State Management**: While the actor model simplifies concurrency by avoiding shared state, managing the overall system state (e.g., tracking ongoing conversations, task progress across multiple agents) becomes more complex. You need robust mechanisms for message acknowledgment, error handling, and potentially distributed logging.
4.  **Fault Tolerance**: A key benefit is improved fault tolerance. If the `RemoteAssistant` worker process crashes, the main process can detect this (e.g., via `is_alive()` or a timeout on `queue.get()`) and potentially restart the worker or re-route the task to another available agent. This is harder to achieve with agents running in the same process.
5.  **Scalability**: This model allows for horizontal scaling. You could launch multiple `remote_assistant_worker` processes, potentially on different machines, and distribute tasks among them using a load-balancing mechanism or a more sophisticated message broker (like RabbitMQ or Kafka).

#### Typical Use Cases for Distributed Agents:

*   **Parallel Task Execution**: Running multiple independent AI tasks concurrently, such as processing a batch of documents, analyzing different data streams, or generating multiple creative outputs in parallel.
*   **Specialized Services**: Deploying agents with unique capabilities (e.g., a code execution agent, a data analysis agent, a creative writing agent) as independent microservices that other agents can call upon.
*   **Resource Isolation**: Allocating dedicated computational resources (GPUs, specific libraries) to certain agents by running them in isolated environments.
*   **Long-Running Processes**: Handling tasks that might take a long time to complete without blocking the main application thread.
*   **Multi-Tenant Systems**: Serving multiple users or applications with dedicated agent instances, ensuring isolation and preventing interference.
*   **Edge Computing**: Deploying lightweight agents on edge devices while coordinating with more powerful agents in the cloud.

For truly large-scale distributed AI agent systems, you would typically integrate AutoGen with specialized distributed computing frameworks like **Ray**, **Dask**, or container orchestration platforms like **Kubernetes**. These frameworks provide advanced features for resource management, fault tolerance, scheduling, and inter-process communication that go beyond what `multiprocessing` offers out-of-the-box.


### Resources

*   **AutoGen Documentation**: Explore advanced topics on agent configuration, custom agents, and group chat management. [AutoGen GitHub](https://github.com/microsoft/autogen)
*   **Python `multiprocessing` Module**: Official documentation for Python's process-based parallelism. [Python Docs: multiprocessing](https://docs.python.org/3/library/multiprocessing.html)
*   **The Actor Model**: A foundational paper on the Actor Model by Carl Hewitt, Peter Bishop, and Richard Steiger. [Actors: A Model of Concurrent Computation in Distributed Systems](https://www.cs.utexas.edu/users/browne/CS395F04/papers/actors.pdf)
*   **Ray Distributed Framework**: Learn about a popular open-source framework for building and running distributed applications, often used with AI workloads. [Ray Documentation](https://docs.ray.io/en/latest/)
*   **Building Distributed Systems with Python**: A general overview of patterns and tools for distributed Python applications. [Real Python: Distributed Systems in Python](https://realpython.com/distributed-systems-in-python/)
